# 06 - Rezultati poredjenja modela

Ovaj notebook ne trenira nista. Cita `ai/results/*.json` koje pise
`uv run python -m foodrec.evaluate --model all --summary` i crta figure za rad.

Ceo lanac pre ovoga:

```bash
uv run python -m foodrec.data
uv run python -m foodrec.split --seed 42
uv run python -m foodrec.train --model popularity
uv run python -m foodrec.train --model itemknn
uv run python -m foodrec.train --model ease        # zaseban proces
uv run python -m foodrec.train --model multdae --device auto
uv run python -m foodrec.train --model multvae --device auto
uv run python -m foodrec.train --model neumf   --device auto
uv run python -m foodrec.evaluate --model all --summary
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from foodrec import config
from foodrec.models import DISPLAY_NAMES, MODEL_KEYS

RESULTS = {}
for key in MODEL_KEYS:
    path = config.RESULTS_DIR / f'{key}.json'
    if path.exists():
        RESULTS[key] = json.loads(path.read_text(encoding='utf-8'))

if not RESULTS:
    raise SystemExit(
        'Nema rezultata u ai/results/. Pokrenite ceo lanac iz prve celije '
        '(za to je potreban Food.com dataset u datasets/).'
    )

FIGURES = config.RESULTS_DIR / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

STATS = next(iter(RESULTS.values()))['dataset_stats']
SPLIT = next(iter(RESULTS.values()))['split']
N_ITEMS = int(SPLIT.get('n_items', STATS.get('n_items', 1)))
print(f"modeli: {', '.join(RESULTS)}")
print(f"{STATS.get('n_positives', 0):,} pozitivnih interakcija, "
      f"{STATS.get('n_users', 0):,} korisnika x {N_ITEMS:,} recepata")

## Stil figura

Jedna serija = jedna boja; servirani model (Mult-VAE) je istaknut narandzastom.
Boje su iz proverene kategoricke palete (kontrast i razdvojivost za daltoniste),
mreza je tanka i povucena u pozadinu, a vrednosti su ispisane direktno uz stubice,
pa figure ostaju citljive i u crno-beloj stampi rada.

In [ ]:
# validated categorical slots: blue, orange, aqua
BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
INK, INK_SOFT, GRID = '#0b0b0b', '#52514e', '#dcdbd6'
SERVED = 'multvae'  # the model that ships

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'axes.titlecolor': INK,
    'axes.labelcolor': INK_SOFT,
    'text.color': INK,
    'xtick.color': INK_SOFT,
    'ytick.color': INK_SOFT,
    'axes.edgecolor': GRID,
    'axes.linewidth': 0.6,
    'grid.color': GRID,
    'grid.linewidth': 0.6,
    'legend.frameon': False,
})


def tidy(ax, xgrid=True):
    """Recessive chrome: hairline grid on the value axis only, no box."""
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.grid(axis='x' if xgrid else 'y', linestyle='-', alpha=0.9)
    ax.set_axisbelow(True)
    ax.tick_params(which='both', length=0)  # log minor ticks too
    return ax


def ordered(metric='recall@20', view='weak'):
    """Model keys sorted by a metric, weakest first (bars read bottom-up)."""
    def value(key):
        payload = RESULTS[key].get(view)
        return payload.get(metric, float('nan')) if isinstance(payload, dict) else -1.0
    return sorted(RESULTS, key=value)


def values(keys, metric, view='weak'):
    out = []
    for key in keys:
        payload = RESULTS[key].get(view)
        out.append(payload.get(metric, np.nan) if isinstance(payload, dict) else np.nan)
    return np.asarray(out, dtype=float)

## Tabela rezultata

In [ ]:
COLUMNS = ('recall@10', 'recall@20', 'ndcg@10', 'ndcg@20', 'coverage@20')

for view, title in (('weak', 'WEAK GENERALIZACIJA'), ('strong', 'STRONG GENERALIZACIJA')):
    print(title)
    print(f"{'model':<12}" + ''.join(f'{c:>13}' for c in COLUMNS))
    for key in MODEL_KEYS:
        if key not in RESULTS:
            continue
        payload = RESULTS[key].get(view)
        if not isinstance(payload, dict):
            print(f'{key:<12}' + ''.join(f"{'N/A':>13}" for _ in COLUMNS))
            continue
        print(f'{key:<12}' + ''.join(f"{payload.get(c, float('nan')):>13.4f}" for c in COLUMNS))
    print()

print(f'slucajno rangiranje: Recall@20 = 20 / {N_ITEMS:,} = {20 / N_ITEMS:.4f}')

## Slika 1 - tacnost u primarnom (weak) pogledu

Svaki korisnik je u treningu, 20% njegovih pozitivnih interakcija je izdvojeno za test.
Isprekidana linija je ocekivani Recall@20 slucajnog rangiranja.

In [ ]:
keys = ordered('recall@20', 'weak')
labels = [DISPLAY_NAMES[k] for k in keys]
colors = [ORANGE if k == SERVED else BLUE for k in keys]
positions = np.arange(len(keys))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4), sharey=True)
for ax, metric in zip(axes, ('recall@20', 'ndcg@20'), strict=True):
    data = values(keys, metric, 'weak')
    ax.barh(positions, data, height=0.62, color=colors)
    for y, value in zip(positions, data, strict=True):
        ax.text(value + max(data) * 0.02, y, f'{value:.3f}', va='center', fontsize=8,
                color=INK_SOFT)
    if metric == 'recall@20':
        ax.axvline(20 / N_ITEMS, color=INK_SOFT, linewidth=0.8, linestyle=(0, (4, 3)))
        ax.text(20 / N_ITEMS, -0.72, ' slucajno', fontsize=7.5, color=INK_SOFT,
                va='center')
    ax.set_xlim(0, max(data) * 1.18)
    ax.set_title(metric.replace('recall', 'Recall').replace('ndcg', 'NDCG'), loc='left')
    tidy(ax)

axes[0].set_yticks(positions, labels)
fig.suptitle('Weak generalizacija - servirani model je istaknut', x=0.02, ha='left',
             fontsize=11, color=INK)
fig.tight_layout()
fig.savefig(FIGURES / 'fig1_weak_tacnost.png')
fig.savefig(FIGURES / 'fig1_weak_tacnost.pdf')

## Slika 2 - weak naspram strong generalizacije

Strong pogled meri hladan start: tih 10% korisnika model nikada nije video, pa se
80% njihovih ocena ubacuje kao ulaz. NeuMF ovde nema sta da ponudi - njegova tabela
korisnickih ugradjivanja nema red za nepoznatog korisnika. To je i glavni argument
za izbor Mult-VAE kao serviranog modela.

In [ ]:
keys = ordered('recall@20', 'weak')
labels = [DISPLAY_NAMES[k] for k in keys]
positions = np.arange(len(keys))
weak = values(keys, 'recall@20', 'weak')
strong = values(keys, 'recall@20', 'strong')
height = 0.34

fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.barh(positions + height / 2 + 0.02, weak, height=height, color=BLUE, label='weak')
ax.barh(positions - height / 2 - 0.02, np.nan_to_num(strong), height=height, color=ORANGE,
        label='strong (hladan start)')

limit = np.nanmax(np.concatenate([weak, strong]))
for y, value in zip(positions, weak, strict=True):
    ax.text(value + limit * 0.015, y + height / 2 + 0.02, f'{value:.3f}', va='center',
            fontsize=8, color=INK_SOFT)
for y, value in zip(positions, strong, strict=True):
    text = f'{value:.3f}' if np.isfinite(value) else 'N/A - nema reda za novog korisnika'
    ax.text((value if np.isfinite(value) else 0) + limit * 0.015, y - height / 2 - 0.02,
            text, va='center', fontsize=8, color=INK_SOFT)

ax.set_yticks(positions, labels)
ax.set_xlim(0, limit * 1.42)
ax.set_xlabel('Recall@20')
ax.set_title('Recall@20 po pogledu evaluacije', loc='left', pad=22)
ax.legend(loc='lower right', ncols=2, bbox_to_anchor=(1.0, 1.0))
tidy(ax)
fig.tight_layout()
fig.savefig(FIGURES / 'fig2_weak_vs_strong.png')
fig.savefig(FIGURES / 'fig2_weak_vs_strong.pdf')

## Slika 3 - pokrivenost kataloga

Udeo recepata koji se bar jednom pojave u necijih prvih 20. Popularnost je ovde
najslabija po definiciji: svima nudi istu listu.

In [ ]:
keys = ordered('coverage@20', 'weak')
labels = [DISPLAY_NAMES[k] for k in keys]
colors = [ORANGE if k == SERVED else BLUE for k in keys]
positions = np.arange(len(keys))
data = values(keys, 'coverage@20', 'weak') * 100

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.barh(positions, data, height=0.62, color=colors)
for y, value in zip(positions, data, strict=True):
    ax.text(value + 1.5, y, f'{value:.1f}%', va='center', fontsize=8, color=INK_SOFT)
ax.set_yticks(positions, labels)
ax.set_xlim(0, 108)
ax.set_xlabel('Coverage@20 (%)')
ax.set_title('Pokrivenost kataloga u weak pogledu', loc='left')
tidy(ax)
fig.tight_layout()
fig.savefig(FIGURES / 'fig3_pokrivenost.png')
fig.savefig(FIGURES / 'fig3_pokrivenost.pdf')

## Slika 4 - tok ucenja neuronskih modela

Validacioni NDCG@20 po epohi. Tacka oznacava epohu koju je rano zaustavljanje
izabralo i koja se koristi za finalno treniranje na svim podacima (`--full`).

In [ ]:
neural = [k for k in ('multdae', 'multvae', 'neumf')
          if k in RESULTS and RESULTS[k].get('training_history')]

if not neural:
    print('Nema zabelezenog toka treniranja.')
else:
    fig, ax = plt.subplots(figsize=(7.2, 3.6))
    ends = []
    for key, color in zip(neural, (BLUE, ORANGE, AQUA), strict=False):
        history = RESULTS[key]['training_history']
        epochs = [row['epoch'] for row in history if 'val_ndcg@20' in row]
        scores = [row['val_ndcg@20'] for row in history if 'val_ndcg@20' in row]
        if not epochs:
            continue
        ax.plot(epochs, scores, color=color, linewidth=2, label=DISPLAY_NAMES[key])
        best = RESULTS[key].get('best_epoch')
        if best in epochs:
            index = epochs.index(best)
            ax.plot([best], [scores[index]], marker='o', markersize=7, color=color,
                    markeredgecolor='white', markeredgewidth=2, zorder=3)
        ends.append((scores[-1], max(epochs), DISPLAY_NAMES[key], color))

    # These curves converge, so the end labels are pushed apart before drawing.
    last_epoch = max(end[1] for end in ends)
    span = ax.get_ylim()[1] - ax.get_ylim()[0]
    previous = -np.inf
    for score, _, label, color in sorted(ends):
        y = max(score, previous + 0.075 * span)
        ax.text(last_epoch + 0.6, y, label, color=color, fontsize=8, va='center')
        previous = y
    ax.set_ylim(top=previous + 0.06 * span)  # keep the pushed-up labels inside
    ax.set_xlim(right=last_epoch * 1.22)
    ax.set_xlabel('epoha')
    ax.set_ylabel('validacioni NDCG@20')
    ax.set_title('Rano zaustavljanje po validacionom NDCG@20', loc='left')
    ax.legend(loc='lower right')
    tidy(ax, xgrid=False)
    fig.tight_layout()
    fig.savefig(FIGURES / 'fig4_tok_ucenja.png')
    fig.savefig(FIGURES / 'fig4_tok_ucenja.pdf')

## Slika 5 - cena skorovanja celog kataloga

Logaritamska skala. Mult-VAE skoruje ceo katalog jednim prolazom kroz mrezu;
NeuMF mora da provuce svaki par (korisnik, recept) kroz MLP, pa raste sa katalogom.
Ovo je drugi razlog zasto se servira Mult-VAE.

In [ ]:
keys = sorted(RESULTS, key=lambda k: RESULTS[k].get('score_ms_per_user', 0.0))
labels = [DISPLAY_NAMES[k] for k in keys]
data = np.asarray([max(RESULTS[k].get('score_ms_per_user', 0.0), 1e-4) for k in keys])
colors = [ORANGE if k == 'neumf' else BLUE for k in keys]
positions = np.arange(len(keys))

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.barh(positions, data, height=0.62, color=colors)
for y, value in zip(positions, data, strict=True):
    ax.text(value * 1.15, y, f'{value:.3g} ms', va='center', fontsize=8, color=INK_SOFT)
ax.set_xscale('log')
ax.set_yticks(positions, labels)
ax.set_xlim(data.min() * 0.5, data.max() * 6)
ax.set_xlabel('milisekunde po korisniku (ceo katalog, log skala)')
ax.set_title('Cena skorovanja', loc='left')
tidy(ax)
fig.tight_layout()
fig.savefig(FIGURES / 'fig5_cena_skorovanja.png')
fig.savefig(FIGURES / 'fig5_cena_skorovanja.pdf')

## Slika 6 - front tacnosti i pokrivenosti

Ovo je slika koja nosi odluku o serviranju. Tacnost skoro ne razlikuje modele, a
pokrivenost ih razlikuje dva reda velicine. Model u donjem desnom uglu je tacan ali
svima nudi istu kratku listu; model u gornjem delu koristi ceo katalog. Oznaceni
model je onaj koji se servira.

In [ ]:
keys = sorted(RESULTS, key=lambda k: values([k], 'coverage@20', 'weak')[0])
recall = values(keys, 'recall@20', 'weak')
coverage = values(keys, 'coverage@20', 'weak') * 100

fig, ax = plt.subplots(figsize=(7.6, 4.2))
# Points cluster tightly in x and in log-y, so label offsets alternate above and
# below to keep ItemKNN and EASE from printing on top of each other.
for rank, (key, x, y) in enumerate(zip(keys, recall, coverage, strict=True)):
    served = key == SERVED
    ax.scatter([x], [y], s=150 if served else 95, color=ORANGE if served else BLUE,
               zorder=3, edgecolor='white', linewidth=1.5)
    dy = 9 if rank % 2 == 0 else -17
    ax.annotate(DISPLAY_NAMES[key], (x, y), textcoords='offset points', xytext=(9, dy),
                fontsize=9, color=INK if served else INK_SOFT,
                fontweight='bold' if served else 'normal')

pop_recall = values(['popularity'], 'recall@20', 'weak')[0]
ax.axvline(pop_recall, color=INK_SOFT, linewidth=0.8, linestyle=(0, (4, 3)), zorder=1)

ax.set_yscale('log')
ax.set_ylim(0.07, 400)
ax.set_yticks([0.1, 1, 10, 100], ['0.1', '1', '10', '100'])
ax.set_xlim(min(recall) - 0.0008, max(recall) + 0.0016)
ax.annotate('popularnost', (pop_recall, 260), textcoords='offset points', xytext=(5, 0),
            fontsize=7.5, color=INK_SOFT, va='center')
ax.set_xlabel('Recall@20 (weak)')
ax.set_ylabel('Coverage@20 (%), log skala')
ratio = coverage.max() / coverage.min()
ax.set_title(
    f'Tacnost razlikuje modele za ~18%, pokrivenost za {ratio:.0f}x', loc='left')
tidy(ax, xgrid=False)
ax.grid(axis='x', linestyle='-', alpha=0.9)
fig.tight_layout()
fig.savefig(FIGURES / 'fig6_front_pokrivenosti.png')
fig.savefig(FIGURES / 'fig6_front_pokrivenosti.pdf')

## Provera zdravog razuma

Iste provere koje `foodrec.evaluate` primenjuje: ako neka padne, tabela ne sme u rad.

In [ ]:
from foodrec.metrics import sanity_check

checks = sanity_check(RESULTS, N_ITEMS, strict=False)
if checks['hard']:
    print('GRESKA - ove provere ukazuju na bag:')
    for problem in checks['hard']:
        print('  -', problem)
if checks['soft']:
    print('UPOZORENJE - ocekivanja kalibrisana na gustim skupovima:')
    for problem in checks['soft']:
        print('  -', problem)
if not checks['hard'] and not checks['soft']:
    print('Sve provere prolaze.')
print()
print('Figure su sacuvane u', FIGURES)